In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsforecast import StatsForecast
from statsforecast.models import Naive, HistoricAverage, WindowAverage, SeasonalNaive
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [7]:
users = pd.read_csv('/workspaces/TestModel/Model/Dataset/user_profiles.csv')
users.info()
users.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   user_id            60 non-null     object 
 1   age_enc            60 non-null     int64  
 2   age_label          60 non-null     object 
 3   bmi                60 non-null     float64
 4   bmi_cat            60 non-null     object 
 5   pcos_confirmed     60 non-null     int64  
 6   pcos_phenotype     60 non-null     object 
 7   cycle_enc          60 non-null     int64  
 8   cycle_label        60 non-null     object 
 9   cycle_days         60 non-null     int64  
 10  cycle_irregular    60 non-null     int64  
 11  insulin_resistant  60 non-null     int64  
 12  stress_baseline    60 non-null     float64
dtypes: float64(2), int64(6), object(5)
memory usage: 6.2+ KB


,age_enc,bmi,pcos_confirmed,cycle_enc,cycle_days,cycle_irregular,insulin_resistant,stress_baseline
count,60.000000,60.000000,60.000000,60.000000,60.000000,60.000000,60.000000,60.000000
mean,1.566667,24.130000,0.616667,2.483333,29.333333,0.233333,0.233333,3.013333
std,1.499906,2.522932,0.490301,1.383825,4.340064,0.426522,0.426522,0.833101
min,0.000000,18.300000,0.000000,0.000000,22.000000,0.000000,0.000000,1.600000
25%,0.000000,22.500000,0.000000,1.000000,26.000000,0.000000,0.000000,2.375000
50%,1.000000,24.100000,1.000000,2.000000,27.000000,0.000000,0.000000,3.000000
75%,2.000000,25.925000,1.000000,3.000000,32.000000,0.000000,0.000000,3.800000
max,6.000000,29.400000,1.000000,5.000000,38.000000,1.000000,1.000000,4.400000


In [8]:
users_context = pd.read_csv('/workspaces/TestModel/Model/Dataset/user_daily_context.csv')
users_context.info()
users_context.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5400 entries, 0 to 5399
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   user_id            5400 non-null   object 
 1   date               5400 non-null   object 
 2   cycle_day          5400 non-null   int64  
 3   cycle_phase        5400 non-null   object 
 4   sleep_hours        5400 non-null   float64
 5   sleep_quality      5400 non-null   int64  
 6   energy_level       5400 non-null   float64
 7   cortisol_est       5400 non-null   float64
 8   water_L            5400 non-null   float64
 9   stress_level       5400 non-null   float64
 10  bmi                5400 non-null   float64
 11  bmi_cat            5400 non-null   object 
 12  pcos_confirmed     5400 non-null   int64  
 13  pcos_phenotype     5400 non-null   object 
 14  insulin_resistant  5400 non-null   int64  
 15  age_enc            5400 non-null   int64  
dtypes: float64(6), int64(5),

,cycle_day,sleep_hours,sleep_quality,energy_level,cortisol_est,water_L,stress_level,bmi,pcos_confirmed,insulin_resistant,age_enc
count,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000
mean,15.199259,6.254667,3.683889,2.406019,3.802374,1.850352,3.015981,24.130000,0.616667,0.233333,1.566667
std,8.841000,0.826683,0.823281,1.111605,0.831620,0.311768,0.918495,2.502051,0.486243,0.422992,1.487492
min,1.000000,3.600000,1.000000,1.000000,1.490000,0.700000,1.000000,18.300000,0.000000,0.000000,0.000000
25%,8.000000,5.700000,3.000000,1.500000,3.170000,1.600000,2.300000,22.500000,0.000000,0.000000,0.000000
50%,15.000000,6.200000,4.000000,2.200000,3.750000,1.800000,3.000000,24.100000,1.000000,0.000000,1.000000
75%,22.000000,6.800000,4.000000,3.100000,4.580000,2.000000,3.700000,25.925000,1.000000,0.000000,2.000000
max,38.000000,9.100000,5.000000,5.000000,5.000000,2.900000,5.000000,29.400000,1.000000,1.000000,6.000000


In [11]:
symptoms = pd.read_csv('/workspaces/TestModel/Model/Dataset/symptom_logs.csv')
symptoms.info()
symptoms.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26908 entries, 0 to 26907
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   user_id            26908 non-null  object 
 1   date               26908 non-null  object 
 2   trigger_timestamp  26908 non-null  object 
 3   cycle_day          26908 non-null  int64  
 4   cycle_phase        26908 non-null  object 
 5   symptom_name       26908 non-null  object 
 6   symptom_category   26908 non-null  object 
 7   triggered          26908 non-null  int64  
 8   severity           26908 non-null  int64  
 9   energy_level       26908 non-null  float64
 10  sleep_prev_night   26908 non-null  float64
 11  stress_level       26908 non-null  float64
 12  cortisol_est       26908 non-null  float64
 13  bmi_cat            26908 non-null  object 
 14  pcos_phenotype     26908 non-null  object 
 15  insulin_resistant  26908 non-null  int64  
 16  pcos_confirmed     269

,cycle_day,triggered,severity,energy_level,sleep_prev_night,stress_level,cortisol_est,insulin_resistant,pcos_confirmed
count,26908.000000,26908.000000,26908.000000,26908.000000,26908.000000,26908.000000,26908.000000,26908.000000,26908.000000
mean,15.198380,0.450944,1.408763,2.403694,6.246882,3.023060,3.805615,0.230229,0.615839
std,8.855217,0.497597,1.745793,1.110041,0.823102,0.918005,0.834227,0.420987,0.486405
min,1.000000,0.000000,0.000000,1.000000,3.600000,1.000000,1.490000,0.000000,0.000000
25%,8.000000,0.000000,0.000000,1.500000,5.700000,2.300000,3.180000,0.000000,0.000000
50%,15.000000,0.000000,0.000000,2.200000,6.200000,3.000000,3.760000,0.000000,1.000000
75%,22.000000,1.000000,3.000000,3.100000,6.800000,3.700000,4.590000,0.000000,1.000000
max,38.000000,1.000000,5.000000,5.000000,9.100000,5.000000,5.000000,1.000000,1.000000


In [12]:
diet = pd.read_csv('/workspaces/TestModel/Model/Dataset/diet_logs.csv')
diet.info()
diet.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15792 entries, 0 to 15791
Data columns (total 49 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_id                15792 non-null  object 
 1   date                   15792 non-null  object 
 2   cycle_phase            15792 non-null  object 
 3   energy_level           15792 non-null  float64
 4   meal_type              15792 non-null  object 
 5   food_name              15792 non-null  object 
 6   protein_g              15792 non-null  float64
 7   carbs_g                15792 non-null  float64
 8   fats_g                 15792 non-null  float64
 9   fiber_g                15792 non-null  float64
 10  calories               15792 non-null  int64  
 11  glycemic_index         15792 non-null  int64  
 12  glycemic_load          15792 non-null  float64
 13  impact_tags_raw        15792 non-null  object 
 14  tag_androgenBoosting   15792 non-null  int64  
 15  ta

,energy_level,protein_g,carbs_g,fats_g,fiber_g,calories,glycemic_index,glycemic_load,tag_androgenBoosting,tag_androgenLowering,tag_antiInflammatory,tag_bloatingTrigger,tag_caffeine,tag_crampReducer,tag_crampTrigger,tag_dairySensitive,tag_energyBoost,tag_gasForming,tag_glutenSensitive,tag_gutFriendly,tag_healthyFats,tag_highCarb,tag_highFibre,tag_highGlycemic,tag_highProtein,tag_insulinBalancing,tag_insulinSpiking,tag_lowCarb,tag_lowFibre,tag_lowGlycemic,tag_mediumGlycemic,tag_moodBoost,tag_noAddedSugar,tag_pcosFriendly,tag_pcosTrigger,tag_periodPainReducer,tag_periodPainTrigger,tag_proInflammatory,tag_processed,tag_sugary,tag_ultraProcessed,tag_unhealthyFats,tag_wholeFood
count,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000,15792.000000
mean,2.565881,9.360119,22.187247,6.251203,2.918693,180.790274,30.299075,9.700861,0.029509,0.263298,0.504179,0.270263,0.115945,0.267351,0.089096,0.064400,0.058067,0.118921,0.064400,0.207067,0.206814,0.027989,0.294643,0.238665,0.349037,0.267984,0.298252,0.055788,0.027989,0.258865,0.059904,0.153116,0.374683,0.614172,0.327761,0.119618,0.029509,0.120187,0.119744,0.115185,0.090679,0.089096,0.495124
std,1.135755,9.388755,19.415905,6.464051,4.373943,115.237800,28.838074,11.553754,0.169233,0.440437,0.499998,0.444110,0.320169,0.442591,0.284891,0.245471,0.233878,0.323706,0.245471,0.405217,0.405033,0.164946,0.455896,0.426281,0.476681,0.442923,0.457505,0.229519,0.164946,0.438025,0.237316,0.360110,0.484056,0.486806,0.469412,0.324524,0.169233,0.325191,0.324672,0.319255,0.287161,0.284891,0.499992
min,1.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.600000,1.500000,0.000000,0.500000,0.000000,110.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2.500000,7.000000,22.000000,3.500000,1.000000,165.000000,25.000000,4.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,3.400000,17.000000,40.000000,10.000000,5.000000,220.000000,60.000000,20.000000,0.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,5.000000,31.000000,58.000000,21.000000,16.000000,450.000000,83.000000,33.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00

In [13]:
activity = pd.read_csv('/workspaces/TestModel/Model/Dataset/activity_logs.csv')
activity.info()
activity.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5400 entries, 0 to 5399
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_id                5400 non-null   object 
 1   date                   5400 non-null   object 
 2   cycle_phase            5400 non-null   object 
 3   cycle_day              5400 non-null   int64  
 4   energy_pre             5400 non-null   float64
 5   effective_energy       5400 non-null   float64
 6   cortisol_pre           5400 non-null   float64
 7   sleep_prev_night       5400 non-null   float64
 8   prev_day_workout       5400 non-null   object 
 9   prev_day_duration      5400 non-null   int64  
 10  prev_day_intensity     5400 non-null   object 
 11  fatigue_carryover      5400 non-null   float64
 12  workout_type           5400 non-null   object 
 13  duration_minutes       5400 non-null   int64  
 14  intensity              5400 non-null   object 
 15  work

,cycle_day,energy_pre,effective_energy,cortisol_pre,sleep_prev_night,prev_day_duration,fatigue_carryover,duration_minutes,workout_completed,phase_appropriateness,fatigue_post,cortisol_post,energy_post,insulin_resistant,pcos_confirmed
count,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000
mean,15.199259,2.406019,2.283778,3.802374,6.254667,16.978148,1.585889,17.169630,0.459815,2.453185,1.586444,3.781659,2.597796,0.233333,0.616667
std,8.841000,1.111605,1.058986,0.831620,0.826683,20.159870,1.376952,20.180289,0.498429,0.893448,1.383351,0.827949,1.021599,0.422992,0.486243
min,1.000000,1.000000,1.000000,1.490000,3.600000,0.000000,0.500000,0.000000,0.000000,0.500000,0.500000,1.000000,1.000000,0.000000,0.000000
25%,8.000000,1.500000,1.400000,3.170000,5.700000,0.000000,0.500000,0.000000,0.000000,2.500000,0.500000,3.180000,1.800000,0.000000,0.000000
50%,15.000000,2.200000,2.100000,3.750000,6.200000,0.000000,0.500000,0.000000,0.000000,2.500000,0.500000,3.780000,2.400000,0.000000,1.000000
75%,22.000000,3.100000,3.000000,4.580000,6.800000,35.000000,2.600000,35.000000,1.000000,3.200000,2.700000,4.500000,3.300000,0.000000,1.000000
max,38.000000,5.000000,5.000000,5.000000,9.100000,60.000000,5.000000,60.000000,1.000000,3.500000,5.000000,5.000000,5.000000,1.000000,1.000000
